In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2025-12-23 20:46:47--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
正在解析主机 raw.githubusercontent.com (raw.githubusercontent.com)... 199.232.68.133
正在连接 raw.githubusercontent.com (raw.githubusercontent.com)|199.232.68.133|:443... 已连接。
已发出 HTTP 请求，正在等待回应... 200 OK
长度：1115394 (1.1M) [text/plain]
正在保存至: “input.txt”

input.txt           100%[===================>]   1.06M  3.32MB/s  用时 0.3s      

2025-12-23 20:46:49 (3.32 MB/s) - 已保存 “input.txt” [1115394/1115394])



In [5]:
with open('input.txt', 'r', encoding="utf-8") as file:
    text = file.read()

In [6]:
print("length:", len(text))

length: 1115394


In [7]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [11]:
chars = sorted(list(set(text)))
vocabSize = len(chars)
print("".join(chars))
print("vocab size:", vocabSize)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65


In [ ]:
stoi = {c:idx for idx, c in enumerate(chars)}
itos = {idx:c for idx, c in enumerate(chars)}
# 这里因为只有65个字符，所以以最简单的方式（enumerate）建立了词表
# python中enumerate函数就是会生成一个可迭代对象，前面是index后面是val
encode = lambda s: [stoi[c] for c in s]
decode = lambda s: [itos[i] for i in s]
# 这里使用匿名函数来快速定义encode和decode。

print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd']


In [15]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [19]:
# 切分训练集和测试集
n = int(0.9 * len(data))
train = data[:n]
validation = data[n:]
print("training set size:   ", len(train))
print("validation set size: ", len(validation))


training set size:    1003854
validation set size:  111540


In [ ]:
blockSize = 8
trainingData = train[:blockSize + 1] # 这里+1是1因为我们不对0context的情况进行预测（这没有意义也做不到。
print(trainingData)

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [28]:
x = train[:blockSize]
y = train[1:blockSize + 1]
for i in range(blockSize):
    context = x[:i + 1]
    nextChar = y[i]
    print(f"When context is {context}, nextChar is {nextChar}")

When context is tensor([18]), nextChar is 47
When context is tensor([18, 47]), nextChar is 56
When context is tensor([18, 47, 56]), nextChar is 57
When context is tensor([18, 47, 56, 57]), nextChar is 58
When context is tensor([18, 47, 56, 57, 58]), nextChar is 1
When context is tensor([18, 47, 56, 57, 58,  1]), nextChar is 15
When context is tensor([18, 47, 56, 57, 58,  1, 15]), nextChar is 47
When context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), nextChar is 58


In [35]:
torch.manual_seed(1112)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train if split == 'train' else validation
    ix = torch.randint(len(data) - block_size, (batch_size, )) # torch.randint(low, high, size) 最后这个元组是输出的形状
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    # torch.stack将一组形状相同的张量拼接成一张大张量
    
    return x, y

xb, yb = get_batch('train')
print("input: ")
print(xb.shape)
print(xb)
print("output: ")
print(yb.shape)
print(yb)

print("-------------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        nextChar = yb[b, t]
        print(f"When context is {context}, nextChar is {nextChar}")

input: 
torch.Size([4, 8])
tensor([[47, 52,  6,  0, 37, 53, 59, 56],
        [40, 56, 43, 39, 58, 46, 11,  0],
        [21,  1, 51, 39, 63,  1, 41, 53],
        [58, 46, 43,  1, 39, 50, 58, 47]])
output: 
torch.Size([4, 8])
tensor([[52,  6,  0, 37, 53, 59, 56,  1],
        [56, 43, 39, 58, 46, 11,  0, 26],
        [ 1, 51, 39, 63,  1, 41, 53, 52],
        [46, 43,  1, 39, 50, 58, 47, 58]])
-------------
When context is tensor([47]), nextChar is 52
When context is tensor([47, 52]), nextChar is 6
When context is tensor([47, 52,  6]), nextChar is 0
When context is tensor([47, 52,  6,  0]), nextChar is 37
When context is tensor([47, 52,  6,  0, 37]), nextChar is 53
When context is tensor([47, 52,  6,  0, 37, 53]), nextChar is 59
When context is tensor([47, 52,  6,  0, 37, 53, 59]), nextChar is 56
When context is tensor([47, 52,  6,  0, 37, 53, 59, 56]), nextChar is 1
When context is tensor([40]), nextChar is 56
When context is tensor([40, 56]), nextChar is 43
When context is tensor([40, 56

In [39]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1112)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, target = None):
        logits = self.token_embedding_table(idx) # (batch, time, channel)

        return logits
    
m = BigramLanguageModel(vocabSize)
out = m(xb, yb) # 这里本质上是在调用forward(nn.Module已经实现了__call__，自动调用forward)
print(out.shape)

torch.Size([4, 8, 65])
